# 向量数据库
## 1. 数据导入


In [ ]:
import os
from openai import OpenAI

In [ ]:

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),  # 如果您没有配置环境变量，请在此处用您的API Key进行替换
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"  # 百炼服务的base_url
)

In [ ]:
completion = client.embeddings.create(
    model="text-embedding-v4",
    input='我想知道迪士尼的退票政策',
    dimensions=1024, # 指定向量维度（仅 text-embedding-v3及 text-embedding-v4支持该参数）
    encoding_format="float"
)

input可以是字符串或字符串列表。

completion：接口返回的结构化响应对象（Pydantic 模型对象）

            执行后，服务端会分析这句话的语义，返回一个长度为 1024 的小数数组，相同语义的句子生成的向量数值会高度接近

In [ ]:

print(completion.model_dump_json()) # .model_dump_json()：把响应对象转为标准 JSON 字符串打印输出

* data:[{""}]
* model:""
* object：""
* usage{"",""}
* id:""

## 2. 数据与元数据一同导入
* 这是一套最简 RAG（检索增强生成）语义检索完整 Demo
* 技术栈：阿里云百炼 text-embedding-v4（文本向量化） + FAISS（本地向量检索库）
* 实现流程：
  * 知识库文本 → 调用 API 转为语义向量 → 存入 FAISS 索引 → 用户提问也转为向量 → FAISS 找出语义最相近文档 → 取出原文用于给大模型做上下文。

简单一句话：用语义相似度查找和用户问题相关的知识库资料，不再局限关键词匹配。

In [ ]:
import os
import numpy as np
import faiss
from openai import OpenAI

In [ ]:
# Step1. 初始化 API 客户端
try:
    client = OpenAI(
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )
except Exception as e:
    print("初始化OpenAI客户端失败，请检查环境变量'DASHSCOPE_API_KEY'是否已设置。")
    print(f"错误信息: {e}")
    exit()

In [ ]:
# Step2. 准备示例文本和元数据
# 在实际应用中，这些数据可能来自数据库、文件等
documents = [
    {
        "id": "doc1",
        "text": "迪士尼乐园的门票一经售出，原则上不予退换。但在特殊情况下，如恶劣天气导致园区关闭，可在官方指引下进行改期或退款。",
        "metadata": {"source": "official_faq_v1.pdf", "category": "退票政策", "author": "Admin"}
    },
    {
        "id": "doc2",
        "text": "购买“奇妙年卡”的用户，可以享受一年内多次入园的特权，并且在餐饮和购物时有折扣。",
        "metadata": {"source": "annual_pass_rules.docx", "category": "会员权益", "author": "MarketingDept"}
    },
    {
        "id": "doc3",
        "text": "对于在线购买的迪士尼门票，如果需要退票，必须在票面日期前48小时通过原购买渠道提交申请，并可能收取手续费。",
        "metadata": {"source": "online_policy.html", "category": "退票政策", "author": "E-commerceTeam"}
    },
    {
        "id": "doc4",
        "text": "园区内的“加勒比海盗”项目因年度维护，将于下周暂停开放。",
        "metadata": {"source": "maintenance_notice.txt", "category": "园区公告", "author": "OpsDept"}
    }
]


In [ ]:
documents

In [ ]:
# Step3. 创建元数据存储和向量列表
# 我们使用一个简单的列表来存储元数据。列表的索引将作为FAISS的ID。
# 这种方式简单直接，适用于中小型数据集。
# 对于大型数据集，可以考虑使用字典或数据库（如Redis, SQLite）
metadata_store = []
vectors_list = []
vector_ids = []
print("正在为文档生成向量...")
for i, doc in enumerate(documents):
    print(i)
    print(doc)
    try:
        # 调用API生成向量
        completion = client.embeddings.create(
            model="text-embedding-v4",
            input=doc["text"],
            dimensions=1024,
            encoding_format="float"
        )
        print(completion.data) # embedding, index, object 以列表的形式存在
        # 获取向量
        vector = completion.data[0].embedding
        vectors_list.append(vector)
        
        # 存储元数据，并使用列表索引作为唯一ID
        metadata_store.append(doc)
        vector_ids.append(i) # 自定义ID与列表索引一致
        
        print(f"  - 已处理文档 {i+1}/{len(documents)}")

    except Exception as e:
        print(f"处理文档 '{doc['id']}' 时出错: {e}")
        continue

# 将向量列表转换为NumPy数组，FAISS需要这种格式
vectors_np = np.array(vectors_list).astype('float32')
vector_ids_np = np.array(vector_ids)


In [ ]:
# Step4. 构建并填充 FAISS 索引
dimension = 1024  # 向量维度
k = 2             # 查找最近的3个邻居

# 创建一个基础的L2距离索引
index_flat_l2 = faiss.IndexFlatL2(dimension)


IndexFlatL2：底层原始索引，功能只有一件事：存放一堆向量，执行暴力 L2 距离搜索。

❌ 原生缺陷：
它不支持自定义 ID。

向量放进去之后，内部自动分配序号：0,1,2,3...。

搜索返回的结果只能是这个内部自增下标，你没法自己指定业务 ID。

举个例子：你想给向量绑定业务 id doc1、doc2，原生 IndexFlatL2 做不到。

In [ ]:
index_flat_l2

In [ ]:

# 使用IndexIDMap来包装基础索引，能够映射我们自定义的ID
# 这就是关联向量和元数据的关键！
index = faiss.IndexIDMap(index_flat_l2)

IndexIDMap包装器：不改动底层检索逻辑，在外层增加 ID 映射能力，给原始索引新增功能。

In [ ]:
index

In [ ]:

# 将向量和它们对应的ID添加到索引中
index.add_with_ids(vectors_np, vector_ids_np)

print(f"\nFAISS 索引已成功创建，共包含 {index.ntotal} 个向量。")


`query_vector = np.array([query_completion.data[0].embedding]).astype('float32')`

导入 numpy，把 Python 原生 list 转换成Numpy 数组

FAISS 底层 C++ 代码只识别 Numpy 数组，不能直接用 Python 列表做检索。

接口返回的向量是 Python float，对应 numpy float64（8 字节）

FAISS 向量检索统一使用 float32（4 字节）

不转换会出现：内存翻倍、检索速度变慢、部分索引报错

In [ ]:
# Step5. 执行搜索并检索元数据
query_text = "我想了解一下迪士尼门票的退款流程"
print(f"\n正在为查询文本生成向量: '{query_text}'")

try:
    # 为查询文本生成向量
    query_completion = client.embeddings.create(
        model="text-embedding-v4",
        input=query_text,
        dimensions=1024,
        encoding_format="float"
    )
    query_vector = np.array([query_completion.data[0].embedding]).astype('float32')

    # 在FAISS索引中执行搜索
    # search方法返回两个NumPy数组：
    # D: 距离 (distances)
    # I: 索引/ID (indices/IDs)
    distances, retrieved_ids = index.search(query_vector, k)
    
    # Step6. 展示结果
    print("\n--- 搜索结果 ---")
    # `retrieved_ids[0]` 包含与查询最相似的k个向量的ID
    for i in range(k):
        doc_id = retrieved_ids[0][i]
        
        # 检查ID是否有效
        if doc_id == -1:
            print(f"\n排名 {i+1}: 未找到更多结果。")
            continue

        # 使用ID从我们的元数据存储中检索信息
        retrieved_doc = metadata_store[doc_id]
        
        print(f"\n--- 排名 {i+1} (L2距离: {distances[0][i]:.4f}) ---")
        print(f"ID: {doc_id}")
        print(f"原始文本: {retrieved_doc['text']}")
        print(f"元数据: {retrieved_doc['metadata']}")

except Exception as e:
    print(f"执行搜索时发生错误: {e}")

In [ ]:
print(index.search(query_vector, k))

In [ ]:
query_text2 = '有无什么购卡策略？'
print(f"\n正在为查询文本生成向量: '{query_text2}'")

try:
    # 为查询文本生成向量
    query_completion2 = client.embeddings.create(
        model="text-embedding-v4",
        input=query_text2,
        dimensions=1024,
        encoding_format="float"
    )
    query_vector2 = np.array([query_completion2.data[0].embedding]).astype('float32')

    # 在FAISS索引中执行搜索
    # search方法返回两个NumPy数组：
    # D: 距离 (distances)
    # I: 索引/ID (indices/IDs)
    distances2, retrieved_ids2 = index.search(query_vector2, k)
    
    # Step6. 展示结果
    print("\n--- 搜索结果 ---")
    # `retrieved_ids[0]` 包含与查询最相似的k个向量的ID
    for i in range(k):
        doc_id = retrieved_ids2[0][i]
        
        # 检查ID是否有效
        if doc_id == -1:
            print(f"\n排名 {i+1}: 未找到更多结果。")
            continue

        # 使用ID从我们的元数据存储中检索信息
        retrieved_doc2 = metadata_store[doc_id]
        
        print(f"\n--- 排名 {i+1} (L2距离: {distances2[0][i]:.4f}) ---")
        print(f"ID: {doc_id}")
        print(f"原始文本: {retrieved_doc2['text']}")
        print(f"元数据: {retrieved_doc2['metadata']}")

except Exception as e:
    print(f"执行搜索时发生错误: {e}")

你用的是 IndexFlatL2（L2 欧氏距离）：
距离数值越小 = 两个向量语义越相似；数值越大 = 语义差距越大。

## 业务 ID doc1/doc2 作为唯一标识版本

In [ ]:
import os
import numpy as np
import faiss
from openai import OpenAI

# Step1. 初始化 API 客户端
try:
    client = OpenAI(
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )
except Exception as e:
    print("初始化OpenAI客户端失败，请检查环境变量'DASHSCOPE_API_KEY'是否已设置。")
    print(f"错误信息: {e}")
    exit()

In [ ]:

# Step2. 准备示例文本和元数据
documents = [
    {
        "id": "doc1",
        "text": "迪士尼乐园的门票一经售出，原则上不予退换。但在特殊情况下，如恶劣天气导致园区关闭，可在官方指引下进行改期或退款。",
        "metadata": {"source": "official_faq_v1.pdf", "category": "退票政策", "author": "Admin"}
    },
    {
        "id": "doc2",
        "text": "购买“奇妙年卡”的用户，可以享受一年内多次入园的特权，并且在餐饮和购物时有折扣。",
        "metadata": {"source": "annual_pass_rules.docx", "category": "会员权益", "author": "MarketingDept"}
    },
    {
        "id": "doc3",
        "text": "对于在线购买的迪士尼门票，如果需要退票，必须在票面日期前48小时通过原购买渠道提交申请，并可能收取手续费。",
        "metadata": {"source": "online_policy.html", "category": "退票政策", "author": "E-commerceTeam"}
    },
    {
        "id": "doc4",
        "text": "园区内的“加勒比海盗”项目因年度维护，将于下周暂停开放。",
        "metadata": {"source": "maintenance_notice.txt", "category": "园区公告", "author": "OpsDept"}
    }
]

In [ ]:

# Step3. 新增ID映射容器 + 向量存储容器
# 双向映射：业务字符串ID <-> FAISS可用整数ID
strid2intid = dict()  # key: doc1/doc2  业务ID  value: FAISS数字ID
intid2strid = dict()  # key: FAISS数字ID value: doc1/doc2 业务ID
metadata_store = dict()  # 元数据字典，key=业务字符串ID(doc1)
vectors_list = []
vector_ids = []  # 存入FAISS的整数ID数组
faiss_auto_id = 1000  # FAISS起始数字ID，从1000开始区分测试下标


In [ ]:

print("正在为文档生成向量...")
for doc in documents:
    business_str_id = doc["id"]  # 业务唯一ID：doc1 doc2 doc3 doc4
    # 分配FAISS专用整数ID
    faiss_int_id = faiss_auto_id
    faiss_auto_id += 1

    # 绑定双向映射关系
    strid2intid[business_str_id] = faiss_int_id
    intid2strid[faiss_int_id] = business_str_id
    metadata_store[business_str_id] = doc  # 元数据按业务ID存储

    try:
        # 调用Embedding接口生成向量
        completion = client.embeddings.create(
            model="text-embedding-v4",
            input=doc["text"],
            dimensions=1024,
            encoding_format="float"
        )
        vector = completion.data[0].embedding
        vectors_list.append(vector)
        vector_ids.append(faiss_int_id)
        print(f"  - 已处理文档 {business_str_id}，分配FAISS整数ID:{faiss_int_id}")
    except Exception as e:
        print(f"处理文档 '{business_str_id}' 时出错: {e}")
        continue

# 转换为FAISS要求的float32 numpy数组
vectors_np = np.array(vectors_list).astype('float32')
vector_ids_np = np.array(vector_ids, dtype="int64")  # FAISS强制int64类型ID

# Step4. 构建FAISS双层索引（逻辑不变）
dimension = 1024
k = 2

# 底层L2索引
index_flat_l2 = faiss.IndexFlatL2(dimension)
# 包装ID映射层，支持自定义整数ID
index = faiss.IndexIDMap(index_flat_l2)
# 写入向量+FAISS整数ID
index.add_with_ids(vectors_np, vector_ids_np)

print(f"\nFAISS 索引已成功创建，共包含 {index.ntotal} 个向量。")

# Step5. 检索示例1：查询退款流程
query_text = "我想了解一下迪士尼门票的退款流程"
print(f"\n正在为查询文本生成向量: '{query_text}'")
try:
    query_completion = client.embeddings.create(
        model="text-embedding-v4",
        input=query_text,
        dimensions=1024,
        encoding_format="float"
    )
    # 构造FAISS标准二维float32查询向量
    query_vector = np.array([query_completion.data[0].embedding]).astype('float32')
    distances, retrieved_ids = index.search(query_vector, k)

    print("\n--- 搜索结果（查询：退款流程） ---")
    for i in range(k):
        faiss_int_id = retrieved_ids[0][i]
        if faiss_int_id == -1:
            print(f"\n排名 {i+1}: 未找到更多结果。")
            continue
        # FAISS数字ID反向转换为业务字符串ID
        business_doc_id = intid2strid[faiss_int_id]
        retrieved_doc = metadata_store[business_doc_id]

        print(f"\n--- 排名 {i+1} (L2距离: {distances[0][i]:.4f}) ---")
        print(f"FAISS内部整数ID: {faiss_int_id}")
        print(f"业务文档ID: {business_doc_id}")
        print(f"原始文本: {retrieved_doc['text']}")
        print(f"元数据: {retrieved_doc['metadata']}")
except Exception as e:
    print(f"执行搜索时发生错误: {e}")

# Step6. 检索示例2：查询购卡策略（你之前的测试query）
query_text2 = '有无什么购卡策略？'
print(f"\n正在为查询文本生成向量: '{query_text2}'")
try:
    query_completion2 = client.embeddings.create(
        model="text-embedding-v4",
        input=query_text2,
        dimensions=1024,
        encoding_format="float"
    )
    query_vector2 = np.array([query_completion2.data[0].embedding]).astype('float32')
    distances2, retrieved_ids2 = index.search(query_vector2, k)

    print("\n--- 搜索结果（查询：购卡策略） ---")
    for i in range(k):
        faiss_int_id = retrieved_ids2[0][i]
        if faiss_int_id == -1:
            print(f"\n排名 {i+1}: 未找到更多结果。")
            continue
        business_doc_id = intid2strid[faiss_int_id]
        retrieved_doc2 = metadata_store[business_doc_id]

        print(f"\n--- 排名 {i+1} (L2距离: {distances2[0][i]:.4f}) ---")
        print(f"FAISS内部整数ID: {faiss_int_id}")
        print(f"业务文档ID: {business_doc_id}")
        print(f"原始文本: {retrieved_doc2['text']}")
        print(f"元数据: {retrieved_doc2['metadata']}")
except Exception as e:
    print(f"执行搜索时发生错误: {e}")

In [ ]:
print(index.search(query_vector2, k))